# 3 – RAG with a Reranker

Retrieval becomes a two-stage process:

| Stage | Goal | Model | Candidates |
|---|---|---|---|
| 1 | **RECALL** | bi-encoder + vector DB | fetch many (top 10) |
| 2 | **PRECISION** | cross-encoder re-scores each candidate | keep the best 4 |

### Why two stages?

**Bi-encoder** (the embedding model from Example 1)

```
question ─> vector ─┐
                    ├─ cosine similarity
chunk    ─> vector ─┘
```

Question and chunk are encoded **separately**. Chunk vectors can be computed ahead of time, so searching millions of chunks is fast – but the model never sees question and chunk together, so subtle relevance gets lost.

**Cross-encoder** (the reranker)

```
[question + chunk] ─> transformer ─> one relevance score
```

Question and chunk are read **together**, so every word of the question can attend to every word of the chunk. Much more accurate, but there is nothing to precompute: one forward pass per (question, chunk) pair. Far too slow for a whole corpus, fine for 10–100 candidates.

So: cast a wide, cheap net first, then let the expensive model sort the catch.

## Imports

New compared with Example 1: `CrossEncoder` from `sentence-transformers`, which loads the reranker model from the Hugging Face Hub and scores (question, chunk) pairs.

In [ ]:
import os
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

## Configuration

- `LLM_MODEL` / `EMBEDDING_MODEL` – the LLM and the bi-encoder for stage 1, both served by Ollama as in Example 1.
- `RERANKER_MODEL` – the cross-encoder for stage 2. This is the one model in these examples that does **not** come from Ollama: Ollama has no reranking API and no reranker models (the request has been open since 2024, [ollama/ollama#3368](https://github.com/ollama/ollama/issues/3368)). So `sentence-transformers` downloads `BAAI/bge-reranker-base` (~1 GB) from the Hugging Face Hub on first use and runs it locally; `cross-encoder/ms-marco-MiniLM-L-6-v2` is a lighter alternative.
- `FETCH_K` – stage 1: how many candidates to pull from the vector DB. Optimise for *recall*: the right chunk only has to be somewhere in this list. Our toy corpus has ~20 chunks; on real data use 20–100.
- `TOP_N` – stage 2: how many survivors after reranking go to the LLM. Optimise for *precision*.

In [ ]:
DATA_DIR = Path("data")

LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")
EMBEDDING_MODEL = "embeddinggemma:300m"
RERANKER_MODEL = "BAAI/bge-reranker-base"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
FETCH_K = 10
TOP_N = 4

print("LLM:", LLM_MODEL)

## Indexing – identical to Example 1

Load, chunk, embed, store. Nothing about the index changes; the reranker only acts at query time. See Example 1 for a step-by-step explanation.

In [ ]:
docs = [
    Document(page_content=path.read_text(encoding="utf-8"),
             metadata={"source": path.name})
    for path in sorted(DATA_DIR.glob("*.txt"))
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} documents -> {len(chunks)} chunks")

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma(
    collection_name="reranker_rag",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_store.reset_collection()
ids = vector_store.add_documents(chunks)
print(f"Indexed {len(ids)} chunks")

## Loading the cross-encoder

`CrossEncoder` downloads the model from the Hugging Face Hub on first use and runs it locally on the CPU (no Ollama involved, see the configuration notes). Its `predict()` method takes a list of `(text_a, text_b)` pairs and returns one score per pair.

In [ ]:
reranker = CrossEncoder(RERANKER_MODEL)

## Stage 1 – RECALL

Plain similarity search as in Example 1, but with a large `k`. It returns `[(Document, cosine_similarity), ...]`, best first.

In [ ]:
def retrieve_candidates(vector_store, question):
    return vector_store.similarity_search_with_relevance_scores(question, k=FETCH_K)

## Stage 2 – PRECISION: rerank with a cross-encoder

We do this by hand so you can see exactly what happens:

1. Build `(question, chunk)` pairs – the cross-encoder reads them jointly.
2. `reranker.predict(pairs)` gives one relevance score per pair (higher = more relevant). These are **not** cosine similarities; only the order matters.
3. Sort by the new score and keep the best `TOP_N`. We also remember each chunk's original position so we can show how far it moved.

LangChain has a ready-made wrapper that does the same in two lines:

```python
from langchain_classic.retrievers import ContextualCompressionRetriever

retriever = ContextualCompressionRetriever(
    base_compressor=<a reranker, e.g. CohereRerank(top_n=TOP_N)>,
    base_retriever=vector_store.as_retriever(search_kwargs={"k": FETCH_K}),
)
```

Hosted rerankers: `CohereRerank` (`langchain-cohere`), `VoyageAIRerank` (`langchain-voyageai`), ... Local ones like ours: any `sentence-transformers` CrossEncoder model.

In [ ]:
def rerank(reranker, question, candidates):
    pairs = [(question, doc.page_content) for doc, _ in candidates]
    scores = reranker.predict(pairs)
    reranked = sorted(
        (
            (doc, float(score), original_rank)
            for original_rank, ((doc, _), score) in enumerate(zip(candidates, scores), 1)
        ),
        key=lambda item: item[1],
        reverse=True,
    )
    return reranked[:TOP_N]

### Showing the effect

`show_comparison` prints the top `TOP_N` according to the vector search next to the top `TOP_N` after reranking, with each chunk's original position in brackets.

In [ ]:
def show_comparison(question, candidates, reranked):
    def preview(doc):
        return doc.page_content[:80].replace("\n", " ")

    print(f"\nQuestion: {question!r}")
    print(f"  Stage 1 - vector search, top {TOP_N} of {FETCH_K}:")
    for rank, (doc, score) in enumerate(candidates[:TOP_N], 1):
        print(f"    {rank}. cos={score:.3f}  {preview(doc)}...")

    print("  Stage 2 - after reranking:")
    for rank, (doc, score, original_rank) in enumerate(reranked, 1):
        moved = f"(was #{original_rank})"
        print(f"    {rank}. rerank={score:.3f} {moved:>10}  {preview(doc)}...")

## GENERATE – identical to Example 1, but fed with the reranked chunks

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the chunks you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def generate(llm, question, reranked):
    context = "\n\n".join(
        f"[{i}] (source: {Path(doc.metadata['source']).name})\n{doc.page_content}"
        for i, (doc, _, _) in enumerate(reranked, 1)
    )
    return (PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

## Run it

The questions are chosen so that several chunks look *topically* similar (all about money, limits, percentages) but only one actually answers. Watch the `(was #n)` column: that is the reranker correcting the vector search.

In [ ]:
QUESTIONS = [
    "Who has to approve an override of an automatically declined loan?",
    "Why are buses less punctual than trams in Harbor City?",
    "How much of Novaterra's 2026 capex goes into grid upgrades?",
]

llm = init_chat_model(LLM_MODEL, temperature=0)
for q in QUESTIONS:
    candidates = retrieve_candidates(vector_store, q)
    reranked = rerank(reranker, q, candidates)
    show_comparison(q, candidates, reranked)
    print("\nAnswer:", generate(llm, q, reranked))
    print("-" * 80)

## Things to try

1. Find a question where the reranker moves a chunk from position 6 or lower into the top 4.
2. Set `TOP_N = 1`. Does the answer survive on a single chunk?
3. Swap in `cross-encoder/ms-marco-MiniLM-L-6-v2` and compare speed and ranking quality.